# WENO Validation — COVO Convergence & Diagnostics

This notebook reads simulation data from `weno_covo_runs/` (generated by `run_weno_covo.sh`) and produces:

1. **Sanity check** — did anything explode? Residual history for each run.
2. **Log-log convergence plot** — L∞ error vs grid spacing with measured slopes.
3. **Centreline v-velocity** — numerical vs exact on the coarsest grid.
4. **Vorticity contours** — side-by-side panels for all schemes.

All plots saved to `weno_covo_runs/plots/`.

**Run all cells in order.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import re

# ──── CONFIGURATION ────
RUNDIR = 'weno_covo_runs'
PLOTDIR = os.path.join(RUNDIR, 'plots')
os.makedirs(PLOTDIR, exist_ok=True)

# Must match run_weno_covo.sh
SCHEMES = ['C6', 'WENO_JS', 'WENO_Z', 'WENO_CU6', 'WENO_CU6M']
GRIDS   = [41, 81, 161]   # add 321 if you ran it
L_DOMAIN = 16.0

# Vortex parameters (must match Preprocessing_routines.f90)
C_VOR = 0.02
R_VOR = 1.0

# Plot styling
STYLES = {
    'C6':       dict(color='black',    ls='-',   marker='s', lw=2.0, label='C6 (baseline)'),
    'WENO_JS':  dict(color='#e41a1c',  ls='--',  marker='o', lw=1.5, label='WENO-JS'),
    'WENO_Z':   dict(color='#377eb8',  ls='--',  marker='^', lw=1.5, label='WENO-Z'),
    'WENO_CU6': dict(color='#4daf4a',  ls='-.',  marker='D', lw=1.5, label='WENO-CU6'),
    'WENO_CU6M':dict(color='#984ea3',  ls='-.',  marker='v', lw=1.5, label='WENO-CU6-M'),
}

# Expected formal orders (for reference slope lines)
FORMAL_ORDER = {
    'C6': 6, 'WENO_JS': 5, 'WENO_Z': 5, 'WENO_CU6': 6, 'WENO_CU6M': 6
}

print(f'Run directory: {os.path.abspath(RUNDIR)}')
print(f'Schemes: {SCHEMES}')
print(f'Grids: {GRIDS}')

## Step 0: Sanity Check — Did Anything Blow Up?

Before looking at errors, check that every run actually finished without NaN or divergence.
Green = ran fine. Red = something went wrong.

In [ ]:
print(f'{"Scheme":<12} {"Grid":<10} {"Status":<12} {"L-inf Error":<16} {"Last Residual"}')
print('─' * 75)

errors = {}  # errors['WENO_Z'] = [err_41, err_81, err_161]

for scheme in SCHEMES:
    errors[scheme] = []
    for ni in GRIDS:
        logfile = os.path.join(RUNDIR, f'{scheme}_N{ni:03d}', 'log.txt')
        err = None
        last_res = None
        status = '\033[91m✗ MISSING\033[0m'
        
        if os.path.exists(logfile):
            with open(logfile) as f:
                lines = f.readlines()
            
            # Check for NaN or crash
            full_text = ''.join(lines)
            if 'NaN' in full_text or 'Infinity' in full_text:
                status = '\033[91m✗ NaN/Inf\033[0m'
            elif 'COVO L-inf' in full_text:
                status = '\033[92m✓ OK\033[0m'
            else:
                status = '\033[93m? No error line\033[0m'
            
            # Extract L-inf error
            for line in lines:
                if 'COVO L-inf' in line:
                    tokens = line.strip().split()
                    try:
                        err = float(tokens[-1])
                    except ValueError:
                        pass
            
            # Extract last residual line (lines with multiple floats)
            for line in reversed(lines):
                parts = line.strip().split()
                if len(parts) >= 5:
                    try:
                        vals = [float(x) for x in parts[:6]]
                        last_res = f'{vals[1]:.2e}'  # first residual component
                        break
                    except (ValueError, IndexError):
                        continue
        
        errors[scheme].append(err)
        err_str = f'{err:.4e}' if err is not None else '—'
        res_str = last_res if last_res else '—'
        print(f'{scheme:<12} {ni}x{ni:<6} {status:<22} {err_str:<16} {res_str}')

print()
# Quick pass/fail
n_ok = sum(1 for s in SCHEMES for e in errors[s] if e is not None)
n_total = len(SCHEMES) * len(GRIDS)
print(f'Runs with valid errors: {n_ok}/{n_total}')
if n_ok == n_total:
    print('\033[92mAll runs completed successfully. Proceed to plots.\033[0m')
else:
    print('\033[93mSome runs missing or failed — plots may be incomplete.\033[0m')

## Plot 1: Log-Log Convergence

The main result. Each scheme should converge at its formal order:
- C6 (compact 6th order): slope ≈ 4–6 (filter limits it on coarse grids)
- WENO-JS: slope ≈ 5 (formally 5th order, degrades at extrema)
- WENO-Z: slope ≈ 5 (fixes the extrema issue)
- WENO-CU6: slope ≈ 6
- WENO-CU6-M: slope ≈ 6

If a WENO slope is ≈ 1 or shows no convergence, something is broken.

In [ ]:
DX_VALS = [L_DOMAIN / (n - 1) for n in GRIDS]

fig, ax = plt.subplots(figsize=(9, 6.5))

for scheme in SCHEMES:
    errs = errors[scheme]
    valid = [(d, e) for d, e in zip(DX_VALS, errs) if e is not None]
    if not valid:
        print(f'  {scheme}: no data, skipping')
        continue
    dxp, ep = zip(*valid)
    s = STYLES[scheme]
    ax.loglog(dxp, ep, color=s['color'], ls=s['ls'], marker=s['marker'],
              label=s['label'], markersize=8, lw=s['lw'])
    
    # Compute and annotate slope
    if len(valid) >= 2:
        log_dx = np.log10(dxp)
        log_e  = np.log10(ep)
        slope = np.polyfit(log_dx, log_e, 1)[0]
        # Place label near the last point
        ax.annotate(f'  slope = {slope:.1f}', xy=(dxp[-1], ep[-1]),
                    fontsize=9, color=s['color'], fontweight='bold',
                    ha='left', va='center')

# Reference slope lines
dx_ref = np.array([DX_VALS[0], DX_VALS[-1]])
for order, ls, lbl in [(2, ':', '2nd'), (4, ':', '4th'), (5, ':', '5th'), (6, ':', '6th')]:
    # Anchor to bottom-right of plot
    y0 = 1e-3  # reference level at coarsest dx
    y_ref = y0 * (dx_ref / dx_ref[0])**order
    ax.loglog(dx_ref, y_ref, color='grey', ls=ls, lw=0.8, alpha=0.5)
    ax.annotate(lbl, xy=(dx_ref[0]*1.05, y0*1.2), fontsize=7, color='grey')

ax.set_xlabel(r'Grid spacing $\Delta x$', fontsize=13)
ax.set_ylabel(r'$L_\infty$ error in $v$', fontsize=13)
ax.set_title('COVO Convergence — WENO vs C6 Baseline', fontsize=14)
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, which='both', alpha=0.3)
ax.set_xlim(DX_VALS[-1]*0.7, DX_VALS[0]*1.5)

fig.tight_layout()
fig.savefig(os.path.join(PLOTDIR, 'WENO_convergence.pdf'), dpi=150)
fig.savefig(os.path.join(PLOTDIR, 'WENO_convergence.png'), dpi=200)
print(f'Saved to {PLOTDIR}/WENO_convergence.pdf')
plt.show()

## Plot 2: Centreline Swirl Velocity

The v-velocity along y = 0 after the vortex has convected one full lap (t = 16).
On a coarse grid, this shows how much each scheme smears the Gaussian peak.

WENO should be slightly more dissipative than C6 on smooth data (the price of shock-capturing),
but CU6/CU6-M should be close to C6.

In [ ]:
N_PLOT = GRIDS[1] if len(GRIDS) > 1 else GRIDS[0]  # use second grid (81)

fig, ax = plt.subplots(figsize=(10, 5))
exact_plotted = False

for scheme in SCHEMES:
    fname = os.path.join(RUNDIR, f'{scheme}_N{N_PLOT:03d}', 'covo_centreline.dat')
    if not os.path.exists(fname):
        print(f'  {scheme}: centreline file not found, skipping')
        continue
    
    data = np.loadtxt(fname, comments='#')
    x = data[:, 0]
    v_num = data[:, 1]
    v_exact = data[:, 2]
    
    if not exact_plotted:
        ax.plot(x, v_exact, 'k-', lw=2.5, label='Exact', zorder=10)
        exact_plotted = True
    
    s = STYLES[scheme]
    ax.plot(x, v_num, color=s['color'], ls=s['ls'], lw=s['lw'], label=s['label'])

ax.set_xlabel('x', fontsize=13)
ax.set_ylabel('v-velocity', fontsize=13)
ax.set_title(f'Centreline Swirl Velocity (y = 0) — {N_PLOT}×{N_PLOT} grid, t = 16', fontsize=13)
ax.legend(fontsize=9, ncol=2)
ax.grid(True, alpha=0.3)

# Zoom inset on the peak
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
ax_inset = inset_axes(ax, width='35%', height='45%', loc='upper right')
for scheme in SCHEMES:
    fname = os.path.join(RUNDIR, f'{scheme}_N{N_PLOT:03d}', 'covo_centreline.dat')
    if not os.path.exists(fname):
        continue
    data = np.loadtxt(fname, comments='#')
    s = STYLES[scheme]
    ax_inset.plot(data[:,0], data[:,1], color=s['color'], ls=s['ls'], lw=s['lw'])

# Plot exact in inset too
if exact_plotted:
    ax_inset.plot(data[:,0], data[:,2], 'k-', lw=2)

# Set inset limits around the peak
ax_inset.set_xlim(-2, 2)
peak_v = C_VOR / (R_VOR**2 * np.e**0.5)  # analytical peak ≈ C * 1/R^2 * exp(-0.5)
ax_inset.set_ylim(-peak_v*0.2, peak_v*1.3)
ax_inset.set_title('Peak zoom', fontsize=8)
ax_inset.tick_params(labelsize=7)
ax_inset.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(os.path.join(PLOTDIR, f'WENO_centreline_N{N_PLOT:03d}.pdf'), dpi=150)
fig.savefig(os.path.join(PLOTDIR, f'WENO_centreline_N{N_PLOT:03d}.png'), dpi=200)
print(f'Saved to {PLOTDIR}/WENO_centreline_N{N_PLOT:03d}.pdf')
plt.show()

## Plot 3: Vorticity Contours

Side-by-side panels showing the vortex shape from each scheme, compared to the exact solution.
A good scheme keeps the contours circular and tight; a bad one distorts them.

In [ ]:
N_PLOT = GRIDS[1] if len(GRIDS) > 1 else GRIDS[0]

# ── Compute exact vorticity ──
x1d = np.linspace(-8, 8, N_PLOT)
y1d = np.linspace(-8, 8, N_PLOT)
X, Y = np.meshgrid(x1d, y1d, indexing='ij')

xc, yc = 0.0, 0.0  # vortex back at origin at t=16
r2 = ((X - xc)**2 + (Y - yc)**2) / R_VOR**2

u_ex = 1.0 - C_VOR * (Y - yc) / R_VOR**2 * np.exp(-r2 / 2.0)
v_ex = C_VOR * (X - xc) / R_VOR**2 * np.exp(-r2 / 2.0)

# Central differences for vorticity (periodic)
dx = L_DOMAIN / (N_PLOT - 1)
dvdx_ex = np.zeros_like(v_ex)
dudy_ex = np.zeros_like(u_ex)
dvdx_ex[1:-1, :] = (v_ex[2:, :] - v_ex[:-2, :]) / (2*dx)
dvdx_ex[0, :]  = (v_ex[1, :] - v_ex[-2, :]) / (2*dx)
dvdx_ex[-1, :] = dvdx_ex[0, :]
dudy_ex[:, 1:-1] = (u_ex[:, 2:] - u_ex[:, :-2]) / (2*dx)
dudy_ex[:, 0]  = (u_ex[:, 1] - u_ex[:, -2]) / (2*dx)
dudy_ex[:, -1] = dudy_ex[:, 0]
vort_exact = np.abs(dvdx_ex - dudy_ex)

# ── Load numerical vorticity fields ──
panels = ['Exact'] + SCHEMES
vort_data = {'Exact': (X, Y, vort_exact)}

for scheme in SCHEMES:
    fname = os.path.join(RUNDIR, f'{scheme}_N{N_PLOT:03d}', 'covo_vorticity.dat')
    if not os.path.exists(fname):
        print(f'  {scheme}: vorticity file not found, skipping')
        continue
    raw = np.loadtxt(fname, comments='#')
    xv = raw[:, 0].reshape(N_PLOT, N_PLOT)
    yv = raw[:, 1].reshape(N_PLOT, N_PLOT)
    wv = raw[:, 2].reshape(N_PLOT, N_PLOT)
    vort_data[scheme] = (xv, yv, wv)

# ── Plot ──
n_panels = len([p for p in panels if p in vort_data])
ncols = min(n_panels, 3)
nrows = (n_panels + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4.5*nrows))
if n_panels == 1:
    axes = np.array([axes])
axes = axes.flatten()

# Contour levels (same for all panels)
vmax = vort_exact.max()
levels = np.linspace(0, vmax * 0.95, 12)

idx = 0
for panel_name in panels:
    if panel_name not in vort_data:
        continue
    ax = axes[idx]
    xp, yp, wp = vort_data[panel_name]
    cs = ax.contourf(xp, yp, wp, levels=levels, cmap='inferno', extend='max')
    ax.contour(xp, yp, wp, levels=levels, colors='white', linewidths=0.3, alpha=0.5)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_aspect('equal')
    ax.set_title(panel_name, fontsize=12, fontweight='bold')
    ax.tick_params(labelsize=8)
    idx += 1

# Hide unused axes
for j in range(idx, len(axes)):
    axes[j].set_visible(False)

fig.suptitle(f'Vorticity Magnitude — {N_PLOT}×{N_PLOT} grid, t = 16', fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(PLOTDIR, f'WENO_vorticity_N{N_PLOT:03d}.pdf'), dpi=150, bbox_inches='tight')
fig.savefig(os.path.join(PLOTDIR, f'WENO_vorticity_N{N_PLOT:03d}.png'), dpi=200, bbox_inches='tight')
print(f'Saved to {PLOTDIR}/WENO_vorticity_N{N_PLOT:03d}.pdf')
plt.show()

## Plot 4: Peak Dissipation Comparison

How much does each scheme smear the vortex peak? This bar chart shows the ratio
of the numerical peak v-velocity to the exact peak, for each scheme on each grid.
1.0 = perfect preservation. Lower = more dissipation.

WENO-JS should be the most dissipative. CU6-M should be closest to C6.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x_positions = np.arange(len(GRIDS))
bar_width = 0.15
offsets = np.arange(len(SCHEMES)) - len(SCHEMES)/2 + 0.5

for idx, scheme in enumerate(SCHEMES):
    ratios = []
    for ni in GRIDS:
        fname = os.path.join(RUNDIR, f'{scheme}_N{ni:03d}', 'covo_centreline.dat')
        if not os.path.exists(fname):
            ratios.append(0)
            continue
        data = np.loadtxt(fname, comments='#')
        v_num = data[:, 1]
        v_exact = data[:, 2]
        peak_exact = np.max(np.abs(v_exact))
        peak_num = np.max(np.abs(v_num))
        ratios.append(peak_num / peak_exact if peak_exact > 0 else 0)
    
    s = STYLES[scheme]
    ax.bar(x_positions + offsets[idx]*bar_width, ratios, bar_width,
           color=s['color'], label=s['label'], alpha=0.85, edgecolor='white')

ax.axhline(y=1.0, color='black', ls='--', lw=1, alpha=0.5, label='Exact')
ax.set_xticks(x_positions)
ax.set_xticklabels([f'{n}×{n}' for n in GRIDS], fontsize=11)
ax.set_xlabel('Grid', fontsize=13)
ax.set_ylabel('Peak v / Exact peak', fontsize=13)
ax.set_title('Vortex Peak Preservation', fontsize=14)
ax.legend(fontsize=9, ncol=3, loc='lower right')
ax.set_ylim(0, 1.15)
ax.grid(True, axis='y', alpha=0.3)

fig.tight_layout()
fig.savefig(os.path.join(PLOTDIR, 'WENO_peak_dissipation.pdf'), dpi=150)
fig.savefig(os.path.join(PLOTDIR, 'WENO_peak_dissipation.png'), dpi=200)
print(f'Saved to {PLOTDIR}/WENO_peak_dissipation.pdf')
plt.show()

## Summary Table

Print a clean table with errors and measured convergence rates.

In [ ]:
DX_VALS = [L_DOMAIN / (n - 1) for n in GRIDS]

print('\n' + '='*85)
print('  WENO COVO CONVERGENCE SUMMARY')
print('='*85)

# Errors
header = f'{"":<6}'
for scheme in SCHEMES:
    header += f'{scheme:>14}'
print(f'\n  L-inf errors:\n  {header}')
print('  ' + '─'*len(header))

for i, ni in enumerate(GRIDS):
    row = f'{ni}x{ni:<4}'
    for scheme in SCHEMES:
        e = errors[scheme][i]
        row += f'{e:>14.4e}' if e is not None else f'{"—":>14}'
    print(f'  {row}')

# Convergence rates
print(f'\n  Measured convergence rates:')
print(f'  {"":<6}', end='')
for scheme in SCHEMES:
    print(f'{scheme:>14}', end='')
print()
print('  ' + '─'*len(header))

for i in range(1, len(GRIDS)):
    label = f'{GRIDS[i-1]}→{GRIDS[i]}'
    row = f'{label:<6}'
    for scheme in SCHEMES:
        e1, e2 = errors[scheme][i-1], errors[scheme][i]
        if e1 is not None and e2 is not None and e1 > 0 and e2 > 0:
            rate = np.log(e1/e2) / np.log(DX_VALS[i-1]/DX_VALS[i])
            row += f'{rate:>14.2f}'
        else:
            row += f'{"—":>14}'
    print(f'  {row}')

# Expected
print(f'\n  {"Expected":<6}', end='')
for scheme in SCHEMES:
    print(f'{FORMAL_ORDER[scheme]:>14}', end='')
print()

print('\n' + '='*85)
print('  Interpretation:')
print('  • If WENO rates match expected orders → implementation is correct')
print('  • JS/Z at ~5, CU6/CU6-M at ~6 → working as intended')
print('  • WENO errors slightly above C6 on smooth data → normal (dissipative by design)')
print('  • WENO errors WAY above C6 or rates ~1 → something is wrong')
print('='*85)

## What to Look For

**If everything works:**
- Convergence slopes match formal orders (5 for JS/Z, 6 for CU6/CU6-M)
- WENO errors are slightly above C6 at all grid sizes (more dissipative, but same ballpark)
- CU6-M errors are closest to C6 (least dissipative WENO variant)
- Vorticity contours stay circular for all schemes
- Peak dissipation: JS smears the most, CU6-M smears the least

**Red flags (something is broken):**
- Slope ≈ 1 or 2 → WENO is falling back to first/second order (weight formula issue?)
- NaN or blow-up → sign convention or alpha computation is wrong
- Error LOWER than C6 → suspicious, check if WENO is actually doing anything
- Vortex drifted to wrong location → metric or wrapping issue

**Next steps after validation:**
1. Run on sinusoidal mesh (set `RUN_CURVILINEAR=1` in the shell script)
2. Test with viscous=1 (TGV case)
3. Sod shock tube to verify shock-capturing